# HCO360 Tables

| Version | Description           |
|---------|----------------------|
| v1 (1/7)     | HCO Enrichment table |

## Purpose
Enriches HCO records with parent organization hierarchies and geographic data.

## Input Tables
- `reference_file` - Base HCO mappings (Julie-mapped NPIs)
- `vod_hco` - VOD parent hierarchy data
- `zip_to_territory_mapping` - Geographic attributes

## Output
- **View**: `reference_file_v5` (temporary)

## Logic Flow

### 1. Base HCO Data
Extract Julie-mapped HCO NPIs with target flags and addresses from reference file.

### 2. Parent Hierarchy Lookup
Join VOD tables to get three parent levels per HCO:
- Hospital parent
- Immediate parent  
- Top parent

### 3. Best Parent Selection
Select single parent using priority order:

top_parent → immediate_parent → hospital_parent → NULL


### 4. Geographic Enrichment
Add city, state, territory, region by joining ZIP-to-territory mapping.

## Key Fields Output
- `hco_npi_old` / `hco_npi_julie` - Old vs Julie-mapped NPIs
- `parent_npi` / `parent_name` - Best available parent
- `julie_npi_city/state/territory/region` - Geographic attributes
- `hco_target` - Target flag

## Technical Notes
- Uses `DISTINCT` to deduplicate at each step
- `TRY_CAST` on ZIP codes (returns NULL for non-numeric)
- Filters VOD data to only Julie-mapped HCO NPIs

In [0]:
/* ============================================================================
   PURPOSE
   ----------------------------------------------------------------------------
   This script builds an HCO “360” enrichment view by:
   1) Pulling HCO attributes from the existing reference_file (Julie-mapped HCO NPIs)
   2) Looking up VOD HCO hierarchy fields (hospital / immediate / top parent)
      for those Julie HCO NPIs
   3) Resolving a single “best available” parent (prefers top → immediate → hospital)
   4) Enriching geo/territory/region attributes from ZIP → territory mapping
   5) Producing a final output with: old vs Julie NPI fields, target flags,
      parent rollups, and ZIP-based geo assignments

   NOTES
   ----------------------------------------------------------------------------
   - Documentation below is aligned to the *actual* flow of CTEs in the query.
   - No SQL logic has been changed; only comments/step labels were added.
   ============================================================================ */

CREATE OR REPLACE TEMPORARY VIEW reference_file_v5 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Pull “HCO 360” base fields from the current reference_file
   - Captures:
     * Old HCO NPI/name (pre-mapping context)
     * Julie-mapped active HCO NPI/name
     * Target flag + Julie-file presence flag
     * HCO address + ZIP
   --------------------------------------------------------------------------- */
hco_360 AS (
  SELECT DISTINCT
      hco_npi_old,
      hco_name_old,
      hco_npi  AS hco_npi_julie,
      hco_name AS hco_name_julie,
      hco_target,
      hco_npi_present_julies_file_flag,
      hco_address,
      hco_zip
  FROM com_edp_prd.cmpa_insights_internal_schema.reference_file
),

/* ---------------------------------------------------------------------------
   STEP 1: Pull hierarchy parent NPIs/names from VOD for the Julie-mapped HCO NPIs
   - "a" is the active entity (Julie NPI) we care about
   - "b/c/d" resolve parent entities at three levels:
       * hospital_parent__v
       * immediate_parent__v
       * top_parent__v
   - Filter: only HCOs present in hco_360 (Julie NPIs)
   --------------------------------------------------------------------------- */
base AS (
  SELECT DISTINCT
      a.npi_num__v        AS active_npi,
      a.corporate_name__v AS active_name,

      -- Hospital parent (lowest parent level in this rollup)
      b.npi_num__v        AS hospital_parent_npi,
      b.corporate_name__v AS hospital_parent_name,

      -- Immediate parent (mid-level)
      c.npi_num__v        AS immediate_parent_npi,
      c.corporate_name__v AS immediate_parent_name,

      -- Top parent (highest-level rollup)
      d.npi_num__v        AS top_parent_npi,
      d.corporate_name__v AS top_parent_name

  FROM com_raw.vod_hco a
  LEFT JOIN com_raw.vod_hco b
      ON a.hospital_parent__v = b.vid__v
  LEFT JOIN com_raw.vod_hco c
      ON a.immediate_parent__v = c.vid__v
  LEFT JOIN com_raw.vod_hco d
      ON a.top_parent__v = d.vid__v

  WHERE a.npi_num__v IN (
      SELECT DISTINCT hco_npi_julie
      FROM hco_360
  )
),

/* ---------------------------------------------------------------------------
   STEP 2: Resolve a single parent NPI/name per HCO (best available parent)
   - Preference order:
       1) top parent
       2) immediate parent
       3) hospital parent
       4) null (no parent found)
   - Join the resolved parent fields back onto the hco_360 rows
   --------------------------------------------------------------------------- */
parent_mapping AS (
  SELECT
      a.*,
      b.parent_npi,
      b.parent_name
  FROM hco_360 AS a
  LEFT JOIN (
      SELECT DISTINCT
          active_npi,
          active_name,

          /* Parent NPI + Name resolved from SAME LEVEL */
          CASE
              WHEN top_parent_npi IS NOT NULL THEN top_parent_npi
              WHEN immediate_parent_npi IS NOT NULL THEN immediate_parent_npi
              WHEN hospital_parent_npi IS NOT NULL THEN hospital_parent_npi
              ELSE NULL
          END AS parent_npi,

          CASE
              WHEN top_parent_npi IS NOT NULL THEN top_parent_name
              WHEN immediate_parent_npi IS NOT NULL THEN immediate_parent_name
              WHEN hospital_parent_npi IS NOT NULL THEN hospital_parent_name
              ELSE NULL
          END AS parent_name
      FROM base
  ) AS b
      ON a.hco_npi_julie = b.active_npi
),

/* ---------------------------------------------------------------------------
   STEP 3: Enrich city/state/territory/region using the HCO ZIP
   - Uses ZIP → territory mapping table
   - Casts hco_zip to BIGINT to align with mapping table datatype
   --------------------------------------------------------------------------- */
city_mapping AS (
  SELECT
      a.*,
      b.city           AS julie_npi_city,
      b.state          AS julie_npi_state,
      b.territory_name AS julie_npi_territory,
      b.region_name    AS julie_npi_region
  FROM parent_mapping AS a
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping AS b
      ON TRY_CAST(a.hco_zip AS BIGINT) = b.zipcode
)

/* ---------------------------------------------------------------------------
   STEP 4: Final output
   --------------------------------------------------------------------------- */
SELECT *
FROM city_mapping;


# HCO 360 DEA License Enrichment

## Purpose
Adds active DEA license number to HCO records.

## Input
- `reference_file_v5` - Base HCO 360 data
- `vod_license` - DEA license records
- `vod_hco` - VID to NPI mapping

## Output
- **View**: `reference_file_v6` (temporary)

## Logic

### 1. Filter Active DEA Licenses
- Entity type = HCO
- License type = DEA
- Status = Active

### 2. Select Best DEA per HCO
Rank by:
1. VALID record state
2. Most recent update date
3. Latest expiration date
4. DEA number (tie-breaker)

Select rank 1 only.

### 3. Map to NPI & Join
Convert VID → NPI, join to base table on `hco_npi_julie`.

## Fields Added
- `dea_number` - Active DEA license (NULL if none)

In [0]:
/* ============================================================================
   PURPOSE
   ----------------------------------------------------------------------------
   This script builds reference_file_v6 by enriching the HCO 360 view (v5) with
   a best-available active DEA license number from VOD for each Julie-mapped HCO.

   High-level flow:
   1) Start from reference_file_v5 as the base dataset
   2) Pull active DEA licenses (VOD license table) for HCO entities
   3) Use deterministic ranking to select ONE “best” DEA record per HCO VID
      (prefers VALID + most recently updated, then latest expiration, etc.)
   4) Map HCO VID → HCO NPI via vod_hco, restrict to NPIs present in v5
   5) Left join DEA number onto the base output

   NOTES
   ----------------------------------------------------------------------------
   - Documentation below matches the CTE flow; no SQL logic changed.
   - DEA is returned as NULL when no qualifying active DEA license exists.
   ============================================================================ */

CREATE OR REPLACE TEMPORARY VIEW reference_file_v6 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Load base dataset (HCO 360 view from v5)
   --------------------------------------------------------------------------- */
base_table AS (
  SELECT *
  FROM reference_file_v5
),

/* ---------------------------------------------------------------------------
   STEP 1: Pull ACTIVE DEA licenses for HCO entities from VOD
   - Filters:
       * entity_type__v = 'HCO'
       * type_value__v  = 'DEA'
       * license_status__v = 'A' (active)
       * license_number__v is present
   - Ranking / selection:
       dense_rank() over entity_vid__v to select one “best” row per HCO VID
       Preference order:
         1) record_state__v = 'VALID' first
         2) most recently updated (status_update_time / modified_date / created_date)
         3) latest expiration_date__v
         4) deterministic tie-breaker on license_number__v
   --------------------------------------------------------------------------- */
dea_table AS (
  SELECT *
  FROM (
      SELECT
          *,
          DENSE_RANK() OVER (
              PARTITION BY entity_vid__v
              ORDER BY
                  CASE WHEN record_state__v = 'VALID' THEN 0 ELSE 1 END,
                  COALESCE(status_update_time__v, modified_date__v, created_date__v) DESC,
                  expiration_date__v DESC,
                  license_number__v ASC
          ) AS pick_rn
      FROM com_raw.vod_license
      WHERE entity_type__v = 'HCO'
        AND type_value__v = 'DEA'
        AND license_status__v = 'A'
        AND license_number__v IS NOT NULL
  )
  WHERE pick_rn = 1
),

/* ---------------------------------------------------------------------------
   STEP 2: Convert HCO VID-level DEA record to NPI-level DEA mapping
   - Join vod_hco (VID → NPI) to the selected DEA license per entity_vid__v
   - Restrict to HCO NPIs present in base_table to limit scope
   --------------------------------------------------------------------------- */
dea_number AS (
  SELECT
      a.npi_num__v          AS npi,
      b.license_number__v   AS dea_number
  FROM com_raw.vod_hco AS a
  LEFT JOIN dea_table AS b
      ON a.vid__v = b.entity_vid__v
  WHERE a.npi_num__v IN (
      SELECT DISTINCT hco_npi_julie
      FROM base_table
  )
)

/* ---------------------------------------------------------------------------
   STEP 3: Final output
   - Left join DEA number onto the base_table using Julie-mapped HCO NPI
   --------------------------------------------------------------------------- */
SELECT
    a.*,
    b.dea_number
FROM base_table AS a
LEFT JOIN dea_number AS b
    ON a.hco_npi_julie = b.npi;
    
CREATE OR REPLACE TEMPORARY VIEW hco_360_base_v1 AS
SELECT * FROM reference_file_v6;


### Adding additional columns

In [0]:
/* =============================================================================
   PURPOSE
   -----------------------------------------------------------------------------
   This script assigns a single “Primary HCP” NPI per eligible patient by:
   1) Building Dx claims (5-year window) using aligned NPI attribution rules
   2) Building Tx claims (5-year window) using corrected NPI attribution rules
   3) Defining eligibility cohorts (Specified + Incremental) using Dx frequency
      and Tx presence in a 2-year window
   4) Combining Dx + Tx claims for eligible patients to form an HCP visit history
   5) Ranking candidate HCPs per patient using a 4-tier method:
        Tier 1: Specialty priority
        Tier 2: Total visit count (Dx + Tx combined)
        Tier 3: Most recent visit date
        Tier 4: NPI tiebreaker (ascending)
   6) Enriching the selected Primary HCP with an affiliated HCO NPI from the
      reference_file mapping

   TIME WINDOWS
   -----------------------------------------------------------------------------
   - Diagnosis window (5 years): 2020-08-01 to 2025-07-31
   - Treatment window (5 years): 2020-08-01 to 2025-07-31
   - Treatment eligibility window (2 years): 2023-08-01 to 2025-07-31

   NPI ATTRIBUTION RULES (aligned with GTM file)
   -----------------------------------------------------------------------------
   - Medical events with NDC:       COALESCE(RENDERING_NPI, REFERRING_NPI)
   - Medical events with procedure: RENDERING_NPI only
   - Pharmacy events:              PRESCRIBER_NPI

   NOTES
   -----------------------------------------------------------------------------
   - This script creates multiple TEMP views used downstream.
   - Step labels below reflect the actual order of object creation.
   ============================================================================= */


/* =============================================================================
   STEP 1: DIAGNOSIS CLAIMS (5yr)
   - Builds a unified Dx claims view across:
     a) Medical events (Dx identified from DIAGNOSIS_CODES; NPI = COALESCE)
     b) Pharmacy events (Dx identified from DIAGNOSIS_CODE; NPI = PRESCRIBER)
   - Output fields standardized to: PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical Events - Dx (COALESCE(RENDERING_NPI, REFERRING_NPI))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

UNION

-- Pharmacy Events - Dx (PRESCRIBER_NPI)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31';


/* =============================================================================
   STEP 2: TREATMENT CLAIMS (5yr) - CORRECTED NPI LOGIC
   - Builds a unified Tx claims view across:
     a) Medical events with NDC codes (NPI = COALESCE)
     b) Medical events with procedure codes (NPI = RENDERING only)
     c) Pharmacy events with NDC codes (NPI = PRESCRIBER)
   - Includes a CODE field to support eligibility logic (e.g., Elaprase)
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical Events - NDC codes (COALESCE(RENDERING_NPI, REFERRING_NPI))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

UNION

-- Medical Events - Procedure codes (RENDERING_NPI only)
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

UNION

-- Pharmacy Events - NDC codes (PRESCRIBER_NPI)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31';


/* =============================================================================
   STEP 3: TREATMENT CLAIMS (2yr subset for eligibility)
   - Restricts Tx claims to the 2-year eligibility window
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    CLAIM_TYPE,
    CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31';


/* =============================================================================
   STEP 4: PATIENT ELIGIBILITY
   -----------------------------------------------------------------------------
   Defines two mutually exclusive eligible cohorts:
   A) Specified:
      - ≥ 2 distinct E761 Dx dates in 5yr window
      - AND any Tx in 2yr window
   B) Incremental:
      - ≥ 2 distinct E763 Dx dates in 5yr window
      - AND Elaprase Tx in 2yr window (subset of Tx codes)
      - AND NOT already Specified
   Output: eligible_patients = specified ∪ incremental
   ============================================================================= */

-- 4A) Specified prerequisite: 2+ E761 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    -- Medical Dx dates for E761
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

    UNION

    -- Pharmacy Dx dates for E761 (paid only)
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4B) Specified patients: 2+ E761 Dx + any Tx in 2yr
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t
    ON e.PATIENT_ID = t.PATIENT_ID;

-- 4C) Incremental prerequisite: 2+ E763 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    -- Medical Dx dates for E763
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

    UNION

    -- Pharmacy Dx dates for E763 (paid only)
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4D) Elaprase Tx in 2yr (subset used for incremental eligibility)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');

-- 4E) Incremental patients: 2+ E763 Dx + Elaprase Tx in 2yr + NOT specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t
    ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- 4F) All eligible patients = specified ∪ incremental
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


/* =============================================================================
   STEP 5: COMBINE Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
   - Builds the visit history that will drive primary HCP ranking
   - Note: Visit counting uses DISTINCT FILL_DATE per patient/NPI (Dx + Tx combined)
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

-- Dx claims (5yr) for eligible patients
SELECT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    CLAIM_TYPE
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

-- Tx claims (5yr) for eligible patients
SELECT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    CLAIM_TYPE
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);


/* =============================================================================
   STEP 6: PRIMARY HCP ASSIGNMENT (4-tier ranking)
   -----------------------------------------------------------------------------
   6A) Build per-patient/per-NPI metrics:
       - Specialty bucket + specialty priority
       - Total visits (Dx + Tx combined) for ranking
       - Dx-only and Tx-only counts (reference)
       - Most recent visit date for ranking
   6B) Rank NPIs per patient using:
       Tier 1: specialty priority (ASC)
       Tier 2: total visits (DESC)
       Tier 3: most recent visit (DESC)
       Tier 4: NPI (ASC)
   6C) Select top-ranked HCP as PRIMARY_HCP_NPI
   6D) Enrich primary HCP with affiliated HCO NPI from reference_file
   ============================================================================= */
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH

/* ---------------------------------------------------------------------------
   6A: HCP metrics per patient/NPI (Dx + Tx combined)
   --------------------------------------------------------------------------- */
hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty Classification (human-readable grouping)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Specialty Priority (Tier 1: lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: Visit count uses Dx + Tx combined (distinct visit dates)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference counts (not used for ranking tiers)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: Most recent visit
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),

/* ---------------------------------------------------------------------------
   6B: Rank candidate HCPs per patient using 4-tier ordering
   --------------------------------------------------------------------------- */
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,  -- Tier 1: Specialty
                NO_OF_VISITS DESC,       -- Tier 2: Total visits (Dx + Tx)
                MOST_RECENT_VISIT DESC,  -- Tier 3: Most recent visit
                NPI ASC                  -- Tier 4: deterministic tiebreaker
        ) AS HCP_RANK
    FROM hcp_metrics
),

/* ---------------------------------------------------------------------------
   6C: Select the top-ranked HCP as the Primary HCP for each patient
   --------------------------------------------------------------------------- */
primary_hcp AS (
    SELECT
        PATIENT_ID,
        NPI AS PRIMARY_HCP_NPI
    FROM ranked_hcps
    WHERE HCP_RANK = 1
),

/* ---------------------------------------------------------------------------
   6D: Enrich Primary HCP with affiliated HCO NPI
   - Uses reference_file mapping (HCP NPI → HCO NPI)
   --------------------------------------------------------------------------- */
enriching_with_hco AS (
    SELECT
        a.*,
        b.hco_npi
    FROM primary_hcp AS a
    LEFT JOIN cmpa_insights_internal_schema.reference_file AS b
        ON a.primary_hcp_npi = b.hcp_npi
)

/* ---------------------------------------------------------------------------
   6E: Final output for primary_hcp view
   --------------------------------------------------------------------------- */
SELECT *
FROM enriching_with_hco;


/* -- OPTIONAL: Persist the view into a physical table (commented out)
   CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
   SELECT * FROM primary_hcp_assignment;
*/


In [0]:
/* ============================================================================
   PURPOSE
   ----------------------------------------------------------------------------
   This script builds hco_360_base_v2 by enriching hco_360_base_v1 with:
   1) 340B eligibility status from VOD (HCO-level attribute)
   2) “Original target” flag based on an approved HCO NPI list (plus primary-mapped
      NPIs via secondary_to_primary_npi)
   3) Counts of affiliated target HCPs (field target list) per HCO
   4) Counts of affiliated MSL target HCPs (explicit NPI list) per HCO
   5) Count of Elaprase patients per HCO using the primary_hcp assignment output
   6) Producing a final, wide HCO 360 output for downstream analytics

   NOTES
   ----------------------------------------------------------------------------
   - Documentation below matches the CTE flow; no SQL logic changed.
   - All joins are LEFT JOINs to preserve the full base_table population.
   ============================================================================ */

CREATE OR REPLACE TEMPORARY VIEW hco_360_base_v2 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Load base dataset (from v1)
   --------------------------------------------------------------------------- */
base_table AS (
  SELECT *
  FROM hco_360_base_v1
),

/* ---------------------------------------------------------------------------
   STEP 1: Add 340B eligibility status from VOD HCO table
   - Pulls `340B_eligible__v` for each Julie-mapped HCO NPI
   --------------------------------------------------------------------------- */
adding_340b_status AS (
  SELECT DISTINCT
      a.hco_npi_julie AS hco_npi,
      b.`340B_eligible__v` AS status_340b
  FROM base_table AS a
  LEFT JOIN com_raw.vod_hco AS b
      ON a.hco_npi_julie = b.npi_num__v
),

/* ---------------------------------------------------------------------------
   STEP 2: Flag “original target” HCOs based on approved NPI list
   - Sets original_target = hco_npi_julie when:
       a) hco_npi_julie is explicitly in the approved list, OR
       b) hco_npi_julie is a PRIMARY NPI whose SECONDARY is in the approved list
   - Otherwise sets original_target = '-'
   - Restricts to valid HCO NPIs (hco_npi_julie != '-')
   --------------------------------------------------------------------------- */
original_target AS (
  SELECT DISTINCT
      hco_npi_julie AS hco_npi,
      CASE
          WHEN (
              hco_npi_julie IN (
                  /* Approved target HCO NPI list (explicit) */
                  '1235234535', '1356496772', '1598784555', '1760480503', '1003947599',
                  '1215921457', '1114969169', '1376544320', '1003878539', '1548212988',
                  '1184649345', '1144266024', '1093894131', '1154302727', '1033439732',
                  '1235339227', '1225249865', '1164686879', '1114924834', '1043447253',
                  '1578693321', '1871886366', '1275564098', '1285832634', '1922178789',
                  '1083789630', '1912939703', '1326092404', '1184046187', '1235148594',
                  '1346297843', '1548207640', '1003961251', '1669530069', '1811299944',
                  '1043397292', '1205291994', '1083630073', '1366515488', '1063702785',
                  '1225259039', '1114206422', '1285605444', '1831318856', '1366556227',
                  '1114925567', '1275842007', '1780676650', '1649347469', '1093728743',
                  '1548286172', '1194787218', '1053632463', '1700128592', '1043380637',
                  '1457458556', '1073527388', '1477643690', '1669429577', '1164426896',
                  '1043371826', '1295789907', '1104819366', '1154448769', '1255431987',
                  '1760476659', '1285641514', '1003063280', '1184968588', '1689747552',
                  '1316135270', '1750458485', '1003863168', '1467682708', '1447423959',
                  '1689919599', '1003102781', '1568435477', '1154373843', '1407995186',
                  '1710072798', '1144211301', '1942685920', '1205822236', '1053356352',
                  '1245520386', '1467505073', '1336245828', '1053437871', '1548343510',
                  '1437292927', '1447221056', '1801025481', '1205935012', '1013079896',
                  '1235250663', '1609824010', '1083949382', '1730254681', '1578568481',
                  '1437365186', '1538101688', '1306883228', '1609869916', '1003083445',
                  '1295788677', '1164801627', '1255339867', '1538208210', '1649261462',
                  '1568596765', '1306079330', '1952359986', '1023188851', '1326332289',
                  '1528100427', '1194897595', '1083781892', '1134329923', '1013092691',
                  '1740354851', '1487844015', '1174651079', '1255422416', '1144548322',
                  '1356359798', '1164474235', '1215931068', '1811944101', '1063443455',
                  '1790740777', '1528042884', '1669462420', '1013071653', '1477531580',
                  '1235582925', '1023286184', '1760498588', '1003465097', '1255748059',
                  '1154339588', '1194790055', '1396837951', '1740215219', '1194883306',
                  '1033395397', '1598826406', '1578792271', '1588633184', '1346251428',
                  '1447355771', '1437170685', '1144376807', '1477549756', '1013143213',
                  '1396781795', '1336524214', '1124558929', '1043263080', '1073576740',
                  '1023494473', '1639538861', '1609906247', '1013924182', '1629514070',
                  '1477941375', '1487904546', '1407801640', '1487760906', '1295137404',
                  '1073835567', '1104867167', '1013939750', '1275694184', '1356543995',
                  '1699720086', '1023134657', '1255401519', '1184709057', '1811325103',
                  '1720488646', '1245216183', '1639370059', '1699769901', '1144282583',
                  '1952333460', '1194932814', '1497838494', '1588730360', '1285676544',
                  '1871660407', '1033210323', '1487079141', '1376577247', '1356695266',
                  '1821017880', '1801828421', '1073673737', '1609915164', '1164478442',
                  '1336132323', '1265431829', '1114931342', '1366492977', '1699721985',
                  '1306825997', '1396129524', '1982972899', '1013981554', '1063743490',
                  '1912992553', '1275130650', '1538157508', '1497723407', '1710408265',
                  '1043497019', '1871910968', '1043554967', '1033225875', '1740268846',
                  '1306804794', '1568469997', '1013996560', '1942275805', '1437136066',
                  '1841844099', '1831187764', '1295794162', '1417946021', '1437119310',
                  '1184612764', '1285174649', '1043435902', '1083708226', '1811466881',
                  '1417580838', '1003010992', '1669683512', '1326119967', '1952390643',
                  '1598895690', '1497760987', '1851361778', '1235488867', '1013559921',
                  '1265780621', '1265686000', '1114971819', '1386894392', '1043596125',
                  '1134144090', '1609044478', '1134205032', '1245295757', '1710042429',
                  '1407843345', '1487622296', '1902959653', '1063617280', '1295826485',
                  '1841657947', '1700946605', '1003951195', '1063631943', '1558575746',
                  '1013062769', '1003039488', '1306013552', '1003103391', '1063954113',
                  '1871540237', '1841513611', '1114971660', '1427082957', '1528132610',
                  '1407813660', '1073706263', '1154350262', '1407813603', '1033172390',
                  '1629503057', '1013959972', '1386826303', '1871998963', '1174579155',
                  '1861530776', '1255393245', '1780694372', '1013026020', '1306227764',
                  '1376697482', '1316385164', '1588832869', '1386188837', '1861689168',
                  '1508835828', '1548293954', '1124163902', '1013459908', '1235406455',
                  '1821027079', '1700987328', '1386041457', '1013924372', '1366647968',
                  '1235600834'
              )
              OR hco_npi_julie IN (
                  /* If a listed secondary maps to a primary, include that primary */
                  SELECT DISTINCT primary_npi
                  FROM com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi
                  WHERE secondary_npi IN (
                      /* Same approved target list repeated as secondaries */
                      '1235234535', '1356496772', '1598784555', '1760480503', '1003947599',
                      '1215921457', '1114969169', '1376544320', '1003878539', '1548212988',
                      '1184649345', '1144266024', '1093894131', '1154302727', '1033439732',
                      '1235339227', '1225249865', '1164686879', '1114924834', '1043447253',
                      '1578693321', '1871886366', '1275564098', '1285832634', '1922178789',
                      '1083789630', '1912939703', '1326092404', '1184046187', '1235148594',
                      '1346297843', '1548207640', '1003961251', '1669530069', '1811299944',
                      '1043397292', '1205291994', '1083630073', '1366515488', '1063702785',
                      '1225259039', '1114206422', '1285605444', '1831318856', '1366556227',
                      '1114925567', '1275842007', '1780676650', '1649347469', '1093728743',
                      '1548286172', '1194787218', '1053632463', '1700128592', '1043380637',
                      '1457458556', '1073527388', '1477643690', '1669429577', '1164426896',
                      '1043371826', '1295789907', '1104819366', '1154448769', '1255431987',
                      '1760476659', '1285641514', '1003063280', '1184968588', '1689747552',
                      '1316135270', '1750458485', '1003863168', '1467682708', '1447423959',
                      '1689919599', '1003102781', '1568435477', '1154373843', '1407995186',
                      '1710072798', '1144211301', '1942685920', '1205822236', '1053356352',
                      '1245520386', '1467505073', '1336245828', '1053437871', '1548343510',
                      '1437292927', '1447221056', '1801025481', '1205935012', '1013079896',
                      '1235250663', '1609824010', '1083949382', '1730254681', '1578568481',
                      '1437365186', '1538101688', '1306883228', '1609869916', '1003083445',
                      '1295788677', '1164801627', '1255339867', '1538208210', '1649261462',
                      '1568596765', '1306079330', '1952359986', '1023188851', '1326332289',
                      '1528100427', '1194897595', '1083781892', '1134329923', '1013092691',
                      '1740354851', '1487844015', '1174651079', '1255422416', '1144548322',
                      '1356359798', '1164474235', '1215931068', '1811944101', '1063443455',
                      '1790740777', '1528042884', '1669462420', '1013071653', '1477531580',
                      '1235582925', '1023286184', '1760498588', '1003465097', '1255748059',
                      '1154339588', '1194790055', '1396837951', '1740215219', '1194883306',
                      '1033395397', '1598826406', '1578792271', '1588633184', '1346251428',
                      '1447355771', '1437170685', '1144376807', '1477549756', '1013143213',
                      '1396781795', '1336524214', '1124558929', '1043263080', '1073576740',
                      '1023494473', '1639538861', '1609906247', '1013924182', '1629514070',
                      '1477941375', '1487904546', '1407801640', '1487760906', '1295137404',
                      '1073835567', '1104867167', '1013939750', '1275694184', '1356543995',
                      '1699720086', '1023134657', '1255401519', '1184709057', '1811325103',
                      '1720488646', '1245216183', '1639370059', '1699769901', '1144282583',
                      '1952333460', '1194932814', '1497838494', '1588730360', '1285676544',
                      '1871660407', '1033210323', '1487079141', '1376577247', '1356695266',
                      '1821017880', '1801828421', '1073673737', '1609915164', '1164478442',
                      '1336132323', '1265431829', '1114931342', '1366492977', '1699721985',
                      '1306825997', '1396129524', '1982972899', '1013981554', '1063743490',
                      '1912992553', '1275130650', '1538157508', '1497723407', '1710408265',
                      '1043497019', '1871910968', '1043554967', '1033225875', '1740268846',
                      '1306804794', '1568469997', '1013996560', '1942275805', '1437136066',
                      '1841844099', '1831187764', '1295794162', '1417946021', '1437119310',
                      '1184612764', '1285174649', '1043435902', '1083708226', '1811466881',
                      '1417580838', '1003010992', '1669683512', '1326119967', '1952390643',
                      '1598895690', '1497760987', '1851361778', '1235488867', '1013559921',
                      '1265780621', '1265686000', '1114971819', '1386894392', '1043596125',
                      '1134144090', '1609044478', '1134205032', '1245295757', '1710042429',
                      '1407843345', '1487622296', '1902959653', '1063617280', '1295826485',
                      '1841657947', '1700946605', '1003951195', '1063631943', '1558575746',
                      '1013062769', '1003039488', '1306013552', '1003103391', '1063954113',
                      '1871540237', '1841513611', '1114971660', '1427082957', '1528132610',
                      '1407813660', '1073706263', '1154350262', '1407813603', '1033172390',
                      '1629503057', '1013959972', '1386826303', '1871998963', '1174579155',
                      '1861530776', '1255393245', '1780694372', '1013026020', '1306227764',
                      '1376697482', '1316385164', '1588832869', '1386188837', '1861689168',
                      '1508835828', '1548293954', '1124163902', '1013459908', '1235406455',
                      '1821027079', '1700987328', '1386041457', '1013924372', '1366647968',
                      '1235600834'
                  )
              )
          )
          THEN hco_npi_julie
          ELSE '-'
      END AS original_target
  FROM base_table
  WHERE hco_npi_julie != '-'
),

/* ---------------------------------------------------------------------------
   STEP 3: Count affiliated “field target” HCPs per HCO
   - Uses reference_file where hcp_target != '-' as indicator of field targeting
   - Output: one row per hco_npi with count of distinct hcp_target values
   --------------------------------------------------------------------------- */
target_hcps_affiliated AS (
  SELECT
      hco_npi,
      COUNT(DISTINCT hcp_target) AS count_field_target_hcps
  FROM cmpa_insights_internal_schema.reference_file
  WHERE hcp_target != '-'
  GROUP BY 1
  ORDER BY 2 DESC
),

/* ---------------------------------------------------------------------------
   STEP 4: Count affiliated MSL target HCPs per HCO
   - Restricts to an explicit list of MSL target HCP NPIs
   - Output: one row per hco_npi with count of distinct hcp_npi
   --------------------------------------------------------------------------- */
msl_target_hcps AS (
  SELECT
      hco_npi,
      COUNT(DISTINCT hcp_npi) AS msl_target_hcps_count
  FROM cmpa_insights_internal_schema.reference_file
  WHERE hcp_npi IN (
      /* Explicit MSL target HCP NPI list */
      '1699743088', '1154431567', '1467848366', '1528585833', '1114949617',
      '1861866717', '1942545314', '1255435301', '1104395656', '1770949901',
      '1134534597', '1831455690', '1003203779', '1477999522', '1902012180',
      '1114360997', '1104906445', '1740218296', '1073711966', '1588735005',
      '1780931956', '1124319462', '1215983382', '1457745341', '1205896933',
      '1508914458', '1568624633', '1609003011', '1992790778', '1689651218',
      '1366671380', '1780635839', '1053539015', '1679869143', '1558523845',
      '1104010982', '1114294477', '1699798603', '1871599050', '1942612049',
      '1740400506', '1629050851', '1326085010', '1326420662', '1356394324',
      '1356429518', '1497071963', '1386686467', '1013168673', '1740566561',
      '1407878796', '1790959294', '1508961897', '1548491699', '1154521722',
      '1902826498', '1699983155', '1144498429', '1760877088', '1992854392',
      '1700044336', '1225476930', '1548559974', '1134149495', '1194986554',
      '1639153174', '1447207154', '1730239153', '1710145362', '1386851335',
      '1437152832', '1841486974', '1811277130', '1366406456', '1487958146',
      '1760561336', '1790716595', '1801812128', '1811934490', '1760652267',
      '1225113954', '1992766695', '1497958243', '1346288461', '1407860570',
      '1740202159', '1487698874', '1003985649', '1396065934', '1821032129',
      '1164497483', '1366829152', '1902864465', '1811985989', '1902324528',
      '1144563602', '1720084114', '1124405311', '1265451736', '1619970787',
      '1013242106', '1386960334', '1457330607', '1023121159', '1437211299',
      '1689694119', '1215118625', '1366552812', '1972554897', '1528223682',
      '1659481679', '1417117243', '1295099240', '1407143878', '1356372783',
      '1053502484', '1518047943', '1952327926', '1427270313', '1346340080',
      '1679617930', '1780029504', '1902187859', '1891794525'
  )
    AND hco_npi != '-'
  GROUP BY 1
  ORDER BY 2 DESC
),

/* ---------------------------------------------------------------------------
   STEP 5: Count Elaprase patients per HCO from claims-based primary HCP assignment
   - Uses primary_hcp output (patient → primary HCP → affiliated HCO NPI)
   - Output: one row per hco_npi with distinct patient count
   --------------------------------------------------------------------------- */
elaprase_patients_per_claims AS (
  SELECT
      hco_npi,
      COUNT(DISTINCT patient_id) AS elaprase_patients_per_claims
  FROM primary_hcp
  WHERE hco_npi IS NOT NULL
  GROUP BY 1
  ORDER BY 2 DESC
),

/* ---------------------------------------------------------------------------
   STEP 6: Final assembly of the enriched HCO 360 base view
   - Adds:
       * status_340b
       * original_target
       * count_field_target_hcps
       * msl_target_hcps_count
       * elaprase_patients_per_claims
   - LEFT JOINs preserve the full base_table population
   --------------------------------------------------------------------------- */
pulling_currently_added_columns AS (
  SELECT
      a.*,
      b.status_340b,
      c.original_target,
      d.count_field_target_hcps,
      e.msl_target_hcps_count,
      f.elaprase_patients_per_claims
  FROM base_table AS a
  LEFT JOIN adding_340b_status AS b
      ON a.hco_npi_julie = b.hco_npi
  LEFT JOIN original_target AS c
      ON a.hco_npi_julie = c.hco_npi
  LEFT JOIN target_hcps_affiliated AS d
      ON a.hco_npi_julie = d.hco_npi
  LEFT JOIN msl_target_hcps AS e
      ON a.hco_npi_julie = e.hco_npi
  LEFT JOIN elaprase_patients_per_claims AS f
      ON a.hco_npi_julie = f.hco_npi
)

/* ---------------------------------------------------------------------------
   STEP 7: Final output
   --------------------------------------------------------------------------- */
SELECT *
FROM pulling_currently_added_columns;


In [0]:
CREATE OR REPLACE TEMP VIEW hco_survey_patient_counts AS
WITH survey_responses AS (
  SELECT DISTINCT
    qr.modified_date__v,
    COALESCE(
      CAST(qr.response__v AS STRING),
      CAST(qr.text__v AS STRING),
      CAST(qr.number__v AS STRING)
    ) AS question_response,
    s.name__v AS survey_name,
    st.account__v AS id,
    st.account_display_name__v AS name,
    c.npi__v AS npi
  FROM com_edp_prd.com_intgr.question_response qr
  LEFT JOIN com_edp_prd.com_intgr.survey s
    ON qr.ctrl_survey__v = s.id
  LEFT JOIN com_edp_prd.com_intgr.survey_target st
    ON qr.survey_target__v = st.id
  LEFT JOIN com_edp_prd.com_intgr.customer c
    ON st.account__v = c.id
  WHERE s.name__v IN ('Denali HCP Survey', 'Denali HCO Survey') and qr.survey_question__v in ('VC2000000001024', 'VC2000000001002')
),
survey_base AS (
  SELECT
    sr.survey_name,
    sr.modified_date__v,
    sr.id,
    sr.name,
    sr.npi,
    CAST(sr.question_response AS INT) AS patient_count
  FROM survey_responses sr
  WHERE sr.question_response IS NOT NULL
),
hco_survey as (
  select id as hco_id, name as hco_name, npi as hco_npi, patient_count as hco_patient_count
  from (select a.id, a.name, a.npi, a.modified_date__v, a.patient_count, row_number() over(partition by a.id order by a.modified_date__v desc) as rn
  from survey_base as a
  where a.survey_name = 'Denali HCO Survey')
  where rn = 1
),
hcp_survey as (
  select id as hcp_id, name as hcp_name, npi as hcp_npi, patient_count as hcp_patient_count
  from (select a.id, a.name, a.npi, a.modified_date__v, a.patient_count, row_number() over(partition by a.id order by a.modified_date__v desc) as rn
  from survey_base as a
  where a.survey_name = 'Denali HCP Survey')
  where rn = 1
),
hcp_with_hco_metrics as (
  select a.*, b.hco_npi, b.hco_name
  from hcp_survey as a
  left join cmpa_insights_internal_schema.reference_file as b on a.hcp_npi = b.hcp_npi and b.hcp_npi != '-'
),
hco_with_julies_mapping as (
  select a.hco_id, coalesce(b.primary_npi, a.hco_npi) as hco_npi,
  case when b.primary_npi is not null then b.primary_name else a.hco_name end as hco_name,
  a.hco_patient_count
  from hco_survey as a
  left join com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi as b 
  -- on a.hco_npi = b.secondary_npi
  on cast(a.hco_npi as string) = cast(b.secondary_npi as string)
),
hcp_rolled_up_numbers as (
  select hco_npi, sum(hcp_patient_count) as hcp_patient_count_sum
  from hcp_with_hco_metrics
  where hco_npi is not null
  group by 1 order by 2 desc
),
rolled_up_numbers_comparison as (
  select a.hco_npi, a.hco_name, a.hco_patient_count, b.hcp_patient_count_sum,
  case when b.hcp_patient_count_sum > a.hco_patient_count then b.hcp_patient_count_sum else a.hco_patient_count end as final_patient_count
  from hco_with_julies_mapping as a
  left join hcp_rolled_up_numbers as b on a.hco_npi = b.hco_npi
),
hco_with_final_patient_count as (
  select hco_npi, hco_name, final_patient_count as survey_patient_counts
  from rolled_up_numbers_comparison
)
select * from hco_with_final_patient_count

In [0]:
create or replace temp view hco_360_base_v3 as 
with base_table as (
  select *
  from hco_360_base_v2
),
hco_level_survey_counts as (
  select *
  from hco_survey_patient_counts
),
hco_msl_target as (
  select distinct hco_npi_julie,
  case when ((hco_npi_julie in ('1235234535', '1144266024', '1215921457', '1477643690', '1003961251', '1114969169', '1912939703', '1346297843', '1225249865', '1609824010', '1710072798', '1235582925', '1306079330', '1144548322', '1003947599', '1184649345', '1760480503', '1235339227', '1164686879', '1669429577', '1376544320', '1366515488', '1043447253', '1154302727', '1285605444', '1003878539', '1093894131', '1285832634', '1689747552', '1295789907', '1134329923', '1104819366', '1821017880', '1326332289', '1023188851', '1013143213', '1437119310', '1447423959', '1568596765', '1700128592', '1053328948', '1154339588', '1013924182', '1437365186', '1902959653', '1073835567', '1013559921', '1285174649', '1033439732', '1083630073', '1548212988', '1164426896', '1154448769', '1235148594', '1053632463', '1942685920', '1083789630', '1336245828', '1679597108', '1114925567', '1003063280', '1669530069', '1639370059', '1578593224', '1871886366', '1043260482', '1700946605', '1396882205', '1548343510', '1487622296', '1003010992', '1841657947', '1750458485', '1447355771', '1386894392', '1356359798', '1265686000', '1275564098', '1760476659', '1023049236', '1659863025', '1376656538', '1013939750')) or
  (hco_npi_julie in (select distinct primary_npi from com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi where secondary_npi in ('1235234535', '1144266024', '1215921457', '1477643690', '1003961251', '1114969169', '1912939703', '1346297843', '1225249865', '1609824010', '1710072798', '1235582925', '1306079330', '1144548322', '1003947599', '1184649345', '1760480503', '1235339227', '1164686879', '1669429577', '1376544320', '1366515488', '1043447253', '1154302727', '1285605444', '1003878539', '1093894131', '1285832634', '1689747552', '1295789907', '1134329923', '1104819366', '1821017880', '1326332289', '1023188851', '1013143213', '1437119310', '1447423959', '1568596765', '1700128592', '1053328948', '1154339588', '1013924182', '1437365186', '1902959653', '1073835567', '1013559921', '1285174649', '1033439732', '1083630073', '1548212988', '1164426896', '1154448769', '1235148594', '1053632463', '1942685920', '1083789630', '1336245828', '1679597108', '1114925567', '1003063280', '1669530069', '1639370059', '1578593224', '1871886366', '1043260482', '1700946605', '1396882205', '1548343510', '1487622296', '1003010992', '1841657947', '1750458485', '1447355771', '1386894392', '1356359798', '1265686000', '1275564098', '1760476659', '1023049236', '1659863025', '1376656538', '1013939750'))))
  then 1 else 0 end as is_hco_msl_target
  from hco_360_base_v2
),
hco_engaged AS (
  SELECT DISTINCT
    COALESCE(c.primary_npi, b.npi__v) AS npi
  FROM com_edp_prd.com_raw.vcrm_call2__v AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  LEFT JOIN cmpa_insights_internal_schema.secondary_to_primary_npi AS c
    ON TRY_CAST(b.npi__v AS STRING) = TRY_CAST(c.secondary_npi AS STRING)
  WHERE TRY_CAST(COALESCE(c.primary_npi, b.npi__v) AS STRING) IN (
    SELECT DISTINCT TRY_CAST(hco_npi_julie AS STRING)
    FROM hco_360_base_v2
    WHERE hco_npi_julie <> '-'
  )
),
hco_profiled AS (
  SELECT DISTINCT
    COALESCE(c.primary_npi, b.npi__v) AS npi
  FROM com_intgr.survey_target AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  LEFT JOIN cmpa_insights_internal_schema.secondary_to_primary_npi AS c
    ON TRY_CAST(b.npi__v AS STRING) = TRY_CAST(c.secondary_npi AS STRING)
  WHERE TRY_CAST(COALESCE(c.primary_npi, b.npi__v) AS STRING) IN (
    SELECT DISTINCT TRY_CAST(hco_npi_julie AS STRING)
    FROM hco_360_base_v2
    WHERE hco_npi_julie <> '-'
  )
),
org_and_facility_type as (
  select distinct a.hco_npi_julie, c.name as org_type, d.name as facility_type
  from base_table as a
  left join com_raw.vod_hco as b on a.hco_npi_julie = b.npi_num__v
  left join com_raw.vod_references as c on b.major_class_of_trade__v = c.code and c.reference_type = 'MajorClassofTrade'
  left join com_raw.vod_references as d on b.hco_type__v = d.code and d.reference_type = 'HCOType'
)
select a.*, b.survey_patient_counts, c.is_hco_msl_target,
case when d.npi is not null then 1 else 0 end as is_hco_engaged,
case when e.npi is not null then 1 else 0 end as is_hco_profiled,
f.org_type, f.facility_type
from base_table as a
left join hco_level_survey_counts as b on try_cast(a.hco_npi_julie as string) = try_cast(b.hco_npi as string)
left join hco_msl_target as c on try_cast(a.hco_npi_julie as string) = try_cast(c.hco_npi_julie as string)
left join hco_engaged as d on try_cast(a.hco_npi_julie as string) = try_cast(d.npi as string)
left join hco_profiled as e on try_cast(a.hco_npi_julie as string) = try_cast(e.npi as string)
left join org_and_facility_type as f on try_cast(a.hco_npi_julie as string) = try_cast(f.hco_npi_julie as string)

In [0]:
create or replace table com_edp_prd.cmpa_insights_internal_schema.hco360 as 
select hco_npi_old, hco_name_old, hco_npi_julie as hco_npi_crosswalked, hco_name_julie as hco_name_crosswalked, hco_target, hco_npi_present_julies_file_flag as hco_npi_present_crosswalked_file_flag, hco_address, hco_zip, parent_npi, parent_name, julie_npi_city as crosswalked_npi_city, julie_npi_state as crosswalked_npi_state, julie_npi_territory as crosswalked_npi_territory, julie_npi_region as crosswalked_npi_region, dea_number, status_340b, original_target, count_field_target_hcps, msl_target_hcps_count, elaprase_patients_per_claims, survey_patient_counts, is_hco_msl_target, is_hco_engaged, is_hco_profiled, org_type, facility_type
from hco_360_base_v3

In [0]:
select hco_npi_old, hco_name_old, hco_npi_julie as hco_npi_crosswalked, hco_name_julie as hco_name_crosswalked, hco_target, hco_npi_present_julies_file_flag as hco_npi_present_crosswalked_file_flag, hco_address, hco_zip, parent_npi, parent_name, julie_npi_city as crosswalked_npi_city, julie_npi_state as crosswalked_npi_state, julie_npi_territory as crosswalked_npi_territory, julie_npi_region as crosswalked_npi_region, dea_number, status_340b, original_target, count_field_target_hcps, msl_target_hcps_count, elaprase_patients_per_claims, survey_patient_counts, is_hco_msl_target, is_hco_engaged, is_hco_profiled, org_type, facility_type
from hco_360_base_v3

#### QC

In [0]:
WITH dea AS (
  select *
  from (SELECT
    *,
    dense_rank() OVER (
      PARTITION BY entity_vid__v
      ORDER BY
        CASE WHEN record_state__v = 'VALID' THEN 0 ELSE 1 END,
        COALESCE(status_update_time__v, modified_date__v, created_date__v) DESC,
        expiration_date__v DESC,
        license_number__v ASC
    ) AS pick_rn
  FROM com_raw.vod_license
  WHERE entity_type__v = 'HCO'
    AND type_value__v = 'DEA'
    AND license_status__v = 'A'
    AND license_number__v IS NOT NULL)
  where pick_rn = 1
)
select entity_vid__v, count(distinct license_number__v)
from dea
group by 1 order by 2 desc


In [0]:
with hcos_with_more_than_one_dea as (
  select entity_vid__v
from com_raw.vod_license
where entity_type__v = 'HCO' and type_value__v = 'DEA' and license_status__v = 'A' and license_number__v is not null
group by 1
having count(distinct license_number__v) >= 5
)
select row_number() over(partition by entity_vid__v order by license_number__v asc) as rn, * 
from com_raw.vod_license
where entity_type__v = 'HCO' and type_value__v = 'DEA' and license_status__v = 'A' and license_number__v is not null and entity_vid__v in (select entity_vid__v from hcos_with_more_than_one_dea) and entity_vid__v = '242988858277364738'
order by entity_vid__v asc, rn asc

In [0]:
WITH accounts_reached AS (
  SELECT DISTINCT
    COALESCE(c.primary_npi, b.npi__v) AS npi
  FROM com_intgr.survey_target AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  LEFT JOIN cmpa_insights_internal_schema.secondary_to_primary_npi AS c
    ON TRY_CAST(b.npi__v AS STRING) = TRY_CAST(c.secondary_npi AS STRING)
  WHERE TRY_CAST(COALESCE(c.primary_npi, b.npi__v) AS STRING) IN (
    SELECT DISTINCT TRY_CAST(hco_npi_julie AS STRING)
    FROM hco_360_base_v2
    WHERE hco_npi_julie <> '-'
  )
)
SELECT *
FROM accounts_reached;


In [0]:
select distinct call_channel__v
from com_edp_prd.com_raw.vcrm_call2__v